In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import pandas as pd

In [3]:
data_path = './data/preprocessed/'
# Load preprocessed data
train_dataset = torch.load(data_path+'train_dataset.pth', weights_only=False)
test_dataset = torch.load(data_path+'test_dataset.pth', weights_only=False)
# label_encoder = joblib.load(data_path+'label_encoder.pkl')

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [ ]:
# Get all samples from train_dataset
X_train_all, y_train_all = train_dataset[:]  # or iterate through

# Assuming y is one-hot encoded: [1,0] for Success, [0,1] for Anomaly
# Find indices of success cases
success_indices = (y_train_all.argmax(dim=1) == 1).nonzero().squeeze()

# Extract only success cases
X_success = X_train_all[success_indices]
y_success = y_train_all[success_indices]

In [ ]:
import torch.nn.functional as F

# 1. Normalize your embeddings first
X_success_normalized = [F.normalize(seq, dim=-1) for seq in X_success]

In [ ]:
class NextEventDataset(torch.utils.data.Dataset):
    def __init__(self, sequences, max_len=50):
        """
        sequences: list of tensors, each [seq_len, embed_dim]
        Handles LEFT-padded sequences (zeros at start)
        """
        self.sequences = sequences
        self.max_len = max_len
        self.pairs = []
        
        print(f"Processing {len(sequences)} sequences...")
        
        for seq_idx, seq in enumerate(sequences):
            # Find where actual data starts (first non-zero row)
            # Method 1: Check if any element is non-zero
            non_zero_mask = (seq != 0).any(dim=1)  # [seq_len]
            
            if not non_zero_mask.any():
                print(f"⚠️ Sequence {seq_idx} is all zeros, skipping")
                continue
                
            # Get actual start and end of data
            data_start = non_zero_mask.nonzero()[0, 0].item() if non_zero_mask.any() else 0
            data_end = non_zero_mask.nonzero()[-1, 0].item() + 1 if non_zero_mask.any() else len(seq)
            
            actual_data = seq[data_start:data_end]
            actual_len = len(actual_data)
            
            if actual_len < 2:
                continue  # Need at least 2 events for prediction
                
            # Create (context, target) pairs from ACTUAL data only
            for i in range(1, actual_len):
                context = actual_data[:i]  # All previous actual events
                target = actual_data[i]     # Next actual event
                
                # Truncate if too long
                if len(context) > max_len:
                    context = context[-max_len:]
                
                self.pairs.append((context, target))
                
            if seq_idx < 3:  # Debug first few
                print(f"Seq {seq_idx}: padded_len={len(seq)}, actual_len={actual_len}, pairs={actual_len-1}")
        
        print(f"Created {len(self.pairs)} training pairs")
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        return self.pairs[idx]

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def left_pad_collate(batch):
    """Left-pad sequences (zeros at start)"""
    contexts, targets = zip(*batch)
    
    # Get max length in this batch
    max_len = max(len(ctx) for ctx in contexts)
    embed_dim = contexts[0].shape[-1]
    batch_size = len(contexts)
    
    # Create left-padded batch
    padded = torch.zeros(batch_size, max_len, embed_dim)
    padding_mask = torch.ones(batch_size, max_len, dtype=torch.bool)
    
    for i, ctx in enumerate(contexts):
        ctx_len = len(ctx)
        start_idx = max_len - ctx_len  # Left padding
        padded[i, start_idx:] = ctx
        padding_mask[i, start_idx:] = False
    
    return padded, torch.stack(targets), padding_mask

In [ ]:
dataset = NextEventDataset(X_success_normalized)
next_event_loader = DataLoader(
    dataset, 
    batch_size=64, 
    shuffle=True,
    collate_fn=left_pad_collate
)


In [6]:
import torch
import torch.nn as nn

class NextEventTransformer(nn.Module):
    def __init__(self, embed_dim=384, num_heads=8, num_layers=6,
                 max_seq_len=50, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.max_seq_len = max_seq_len

        # learned positional embeddings
        self.pos_emb = nn.Parameter(
            torch.randn(1, max_seq_len, embed_dim) * 0.02
        )

        # encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=4 * embed_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            layer_norm_eps=1e-6
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # prediction head
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 2 * embed_dim),
            nn.GELU(),
            nn.Linear(2 * embed_dim, embed_dim),
        )

        self._init_weights()

    def _init_weights(self):
        for p in self.encoder.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        for m in self.head:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x, padding_mask=None):
        # x: (B, T, D)
        
        # Use provided mask or detect zeros
        if padding_mask is None:
            padding_mask = (x.sum(dim=-1) == 0)
        
        # Add positional embeddings
        x = x + self.pos_emb[:, :x.size(1)]
        x = x * (self.embed_dim ** 0.5)
        
        # Transformer
        enc = self.encoder(x, src_key_padding_mask=padding_mask)
        
        # Get last non-padding token
        lengths = (~padding_mask).sum(dim=1)
        idx = torch.clamp(lengths - 1, min=0)
        
        # Gather last hidden states
        b = x.size(0)
        last_h = enc[torch.arange(b), idx]
        
        # Predict next embedding
        return self.head(last_h)


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NextEventTransformer(embed_dim=384).to(device)

In [ ]:
print("="*50)
print("QUICK DIAGNOSTICS")
print("="*50)

# Get one batch
batch_X, batch_y, mask = next(iter(next_event_loader))
print(f"Batch X shape: {batch_X.shape}")
print(f"Batch y shape: {batch_y.shape}")
print(f"Mask shape: {mask.shape}")

# Check padding percentage
padding_pct = mask.float().mean()*100
print(f"\nPadding: {padding_pct:.1f}%")

# Check non-padding values (FIXED)
if mask.shape[1] == batch_X.shape[1]:  # Mask matches sequence length
    # Expand mask to match X dimensions
    mask_expanded = mask.unsqueeze(-1).expand_as(batch_X)
    non_pad = batch_X[~mask_expanded].view(-1, batch_X.shape[-1])
    
    print(f"\nNon-padding values:")
    print(f"  Count: {len(non_pad)} vectors")
    if len(non_pad) > 0:
        print(f"  Min: {non_pad.min():.4f}, Max: {non_pad.max():.4f}")
        print(f"  Norms: {torch.norm(non_pad, dim=-1).mean():.4f} (should be ~1.0)")
    else:
        print("  No non-padding values found!")
else:
    print(f"\n⚠️ Mask shape {mask.shape} doesn't match X shape {batch_X.shape}")

# Check targets
print(f"\nTargets:")
print(f"  Norms: {torch.norm(batch_y, dim=-1).mean():.4f} (should be ~1.0)")

with torch.no_grad():
    out = model(batch_X.to(device), mask.to(device))
    out_norm = F.normalize(out, dim=-1)
    target_norm = F.normalize(batch_y.to(device), dim=-1)
    cos_sim = (out_norm * target_norm).sum(-1).mean()
    print(f"\nModel test:")
    print(f"  Output shape: {out.shape}")
    print(f"  Initial cosine sim: {cos_sim:.4f}")
    print(f"  Initial loss: {1-cos_sim:.4f}")

In [ ]:
import torch
import torch.nn.functional as F
import os
import math
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = NextEventTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)

warmup_steps = 500
base_lr = 5e-5
total_steps = len(next_event_loader) * 5
global_step = 0

# Save directory
save_dir = "models/transformer/checkpoints"
os.makedirs(save_dir, exist_ok=True)

# Track best model
best_loss = float('inf')
best_model_state = None
patience = 1  # Stop if loss doesn't improve for 2 epochs
patience_counter = 0

# Training loop
for epoch in range(5):
    model.train()
    pbar = tqdm(next_event_loader, desc=f"Epoch {epoch+1}", ncols=100)
    
    epoch_loss = 0
    num_batches = 0

    for batch_X, batch_y, padding_mask in pbar:
        global_step += 1

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        padding_mask = padding_mask.to(device)

        # ---- LR schedule ----
        if global_step < warmup_steps:
            lr = base_lr * (global_step / warmup_steps)
        else:
            progress = (global_step - warmup_steps) / (total_steps - warmup_steps)
            progress = min(progress, 1.0)
            lr = base_lr * 0.5 * (1 + math.cos(math.pi * progress))

        for g in optimizer.param_groups:
            g["lr"] = lr

        # ---- forward ----
        pred = model(batch_X, padding_mask)
        pred_n = F.normalize(pred, dim=-1)
        tgt_n = F.normalize(batch_y, dim=-1)

        loss = 1 - (pred_n * tgt_n).sum(-1).mean()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1
        
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": lr})

    # Calculate average epoch loss
    avg_epoch_loss = epoch_loss / num_batches
    print(f"\nEpoch {epoch+1} completed. Average loss: {avg_epoch_loss:.4f}")
    
    # Check if this is the best model
    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        best_model_state = model.state_dict().copy()  # Save best weights
        patience_counter = 0  # Reset patience
        
        # Save best model
        torch.save({
            'epoch': epoch,
            'global_step': global_step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
        }, os.path.join(save_dir, "best_model.pth"))
        print(f"✅ New best model! Loss: {best_loss:.4f} (Cosine similarity: {1-best_loss:.4f})")
    else:
        patience_counter += 1
        print(f"⚠️  Loss didn't improve. Patience: {patience_counter}/{patience}")
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\n🚨 Early stopping triggered! No improvement for {patience} epochs.")
        print(f"Best loss: {best_loss:.4f} at epoch {epoch+1-patience}")
        break

# Load the best model weights back
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f"\nLoaded best model with loss: {best_loss:.4f}")

# Save final model (with best weights)
torch.save({
    'epoch': epoch,
    'global_step': global_step,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': best_loss,
}, os.path.join(save_dir, "final_model.pth"))

print(f"\n🏆 Training complete! Best cosine similarity: {1-best_loss:.4f}")
print(f"Models saved to: {save_dir}/")

In [14]:
checkpoint = torch.load("models/transformer/checkpoints/final_model.pth", map_location=device)


In [ ]:
def get_prediction_similarity(model, context, target):
    """Calculate cosine similarity between prediction and target"""
    device = next(model.parameters()).device
    
    # Prepare input
    if len(context) > model.max_seq_len:
        context = context[-model.max_seq_len:]
    
    if len(context) < model.max_seq_len:
        padded = torch.zeros((model.max_seq_len, context.shape[-1]), device=device)
        start_idx = model.max_seq_len - len(context)
        padded[start_idx:] = context.to(device)
        mask = torch.ones(model.max_seq_len, dtype=torch.bool, device=device)
        mask[start_idx:] = False
    else:
        padded = context.to(device)
        mask = torch.zeros(model.max_seq_len, dtype=torch.bool, device=device)
    
    # Predict
    with torch.no_grad():
        pred = model(padded.unsqueeze(0), mask.unsqueeze(0))  # [1, 384]
        pred_n = F.normalize(pred.squeeze(0), dim=-1)  # [384]
        target_n = F.normalize(target.to(device), dim=-1)  # [384]
        
        cos_sim = (pred_n * target_n).sum().item()
    
    return cos_sim

# Try predicting on your ORIGINAL training data
print("\nTesting on ORIGINAL X_success (training data):")
if 'X_success' in locals() and len(X_success) > 0:
    train_seq = X_success[0]
    
    # Use same prediction code
    context = train_seq[:20]  # First 20 events
    target = train_seq[20]    # 21st event
    
    # Calculate similarity
    sim = get_prediction_similarity(model, context, target)
    print(f"Training sequence similarity: {sim:.4f}")
    print(f"Should be ~0.93 like your training loss suggests!")
else:
    print("X_success not found")


Testing on ORIGINAL X_success (training data):
X_success not found


In [ ]:
# def extract_test_sequences_clean(test_dataset, max_sequences=500):
#     """
#     Extract clean sequences from (sequence, label) dataset
#     """
#     test_sequences = []
    
#     print(f"\nExtracting sequences from test_dataset...")
#     print(f"Dataset has {len(test_dataset)} samples")
    
#     for i in range(min(max_sequences, len(test_dataset))):
#         sequence, label = test_dataset[i]
        
#         # Remove left padding (zeros at start)
#         non_zero_mask = (sequence != 0).any(dim=1)
        
#         if non_zero_mask.any():
#             start_idx = non_zero_mask.nonzero()[0, 0].item()
#             clean_seq = sequence[start_idx:]
            
#             # Only keep sequences with reasonable length
#             if len(clean_seq) >= 15:  # Need enough for context + prediction
#                 test_sequences.append(clean_seq)
            
#             # Debug first few
#             if i < 3:
#                 print(f"Sample {i}:")
#                 print(f"  Original: {sequence.shape}, non-zero: {non_zero_mask.sum().item()}/50")
#                 print(f"  Clean: {clean_seq.shape}, start_idx: {start_idx}")
#                 print(f"  Label: {label}")
#         else:
#             print(f"⚠️ Sample {i}: All zeros!")
    
#     print(f"\n✅ Extracted {len(test_sequences)} clean sequences")
    
    
#     return test_sequences

# # Extract sequences
# test_sequences = extract_test_sequences_clean(test_dataset, max_sequences=500)

In [ ]:
# def test_single_step_accuracy_at_all_positions(model, test_sequences, min_context=5):
#     """
#     Test next-event prediction at EVERY position in sequence
#     This is what your model was trained for!
#     """
#     print("\n" + "="*60)
#     print("SINGLE-STEP PREDICTION AT ALL POSITIONS")
#     print("="*60)
    
#     device = next(model.parameters()).device  # Get model's device
#     all_similarities = []
#     position_results = {}  # Track by position in sequence
    
#     for seq_idx, seq in enumerate(test_sequences[:200]):  # Test 100 sequences
#         if len(seq) < min_context + 1:
#             continue
        
#         seq_similarities = []
        
#         # For each position i, predict i+1
#         for i in range(min_context, len(seq) - 1):
#             context = seq[:i+1]  # Events 0 through i
#             target = seq[i+1]    # Event i+1 (what we're predicting)
            
#             # Prepare input (left pad to max_len)
#             if len(context) > model.max_seq_len:
#                 context = context[-model.max_seq_len:]
            
#             if len(context) < model.max_seq_len:
#                 padded = torch.zeros((model.max_seq_len, seq.shape[-1]), device=device)  # Use device
#                 start_idx = model.max_seq_len - len(context)
#                 padded[start_idx:] = context.to(device)  # Move to device
#                 mask = torch.ones(model.max_seq_len, dtype=torch.bool, device=device)
#                 mask[start_idx:] = False
#             else:
#                 padded = context.to(device)  # Move to device
#                 mask = torch.zeros(model.max_seq_len, dtype=torch.bool, device=device)
            
#             # Predict
#             with torch.no_grad():
#                 pred = model(padded.unsqueeze(0), mask.unsqueeze(0))
#                 pred_n = F.normalize(pred.squeeze(0), dim=-1)
#                 target_n = F.normalize(target.to(device), dim=-1)  # Move target to device
                
#                 cos_sim = (pred_n * target_n).sum().item()
                
#             seq_similarities.append(cos_sim)
            
#             # Track by position
#             pos_in_sequence = i - min_context  # Position relative to context
#             if pos_in_sequence not in position_results:
#                 position_results[pos_in_sequence] = []
#             position_results[pos_in_sequence].append(cos_sim)
        
#         all_similarities.extend(seq_similarities)
        
#         # Print first few sequences
#         if seq_idx < 3:
#             print(f"\nSequence {seq_idx} (length {len(seq)}):")
#             print(f"  Average similarity: {np.mean(seq_similarities):.4f}")
#             print(f"  Min: {min(seq_similarities):.4f}, Max: {max(seq_similarities):.4f}")
    
#     # Overall results
#     if all_similarities:
#         all_sims = np.array(all_similarities)
        
#         print(f"\nOVERALL RESULTS:")
#         print(f"  Sequences tested: {len(test_sequences[:100])}")
#         print(f"  Total predictions: {len(all_sims)}")
#         print(f"  Average cosine similarity: {all_sims.mean():.4f}")
#         print(f"  Median cosine similarity: {np.median(all_sims):.4f}")
#         print(f"  Std deviation: {all_sims.std():.4f}")
        
#         # Results by position in sequence
#         print(f"\nPERFORMANCE BY POSITION IN SEQUENCE:")
#         positions = sorted(position_results.keys())
#         for pos in positions[:10]:  # First 10 positions
#             pos_sims = position_results[pos]
#             print(f"  Position {pos+1}: {np.mean(pos_sims):.4f} (n={len(pos_sims)})")
        
#         # Anomaly detection thresholds
#         print(f"\n🚨 ANOMALY DETECTION POTENTIAL:")
#         thresholds = [0.99, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70]
#         for thresh in thresholds:
#             below = np.sum(all_sims < thresh)
#             pct = below / len(all_sims) * 100
#             print(f"  Similarity < {thresh}: {pct:.1f}% ({below} predictions)")
        
#         return all_sims, position_results
    
#     return None, None

# all_similarities, position_results = test_single_step_accuracy_at_all_positions(
#     model, test_sequences, min_context=5
# )

In [ ]:
# import torch.nn.functional as F


# def test_anomaly_accuracy(model, test_dataset, threshold=0.9):
#     """Test if model correctly identifies Fail vs Success sequences"""
    
#     results = []
    
#     for i in range(200):
#         seq, label_vec = test_dataset[i]
        
#         # Skip if too short
#         if len(seq) < 15:
#             continue
        
#         # True label: 0=Fail, 1=Success
#         true_label = 1 if label_vec[1] == 1 else 0
        
#         similarities = []
#         device = next(model.parameters()).device
        
#         for pos in range(5, len(seq) - 1):
#             context = seq[:pos+1]
#             target = seq[pos+1]
            
#             if len(context) > model.max_seq_len:
#                 context = context[-model.max_seq_len:]
            
#             if len(context) < model.max_seq_len:
#                 padded = torch.zeros((model.max_seq_len, seq.shape[-1]), device=device)
#                 start_idx = model.max_seq_len - len(context)
#                 padded[start_idx:] = context.to(device)
#                 mask = torch.ones(model.max_seq_len, dtype=torch.bool, device=device)
#                 mask[start_idx:] = False
#             else:
#                 padded = context.to(device)
#                 mask = torch.zeros(model.max_seq_len, dtype=torch.bool, device=device)
            
#             with torch.no_grad():
#                 pred = model(padded.unsqueeze(0), mask.unsqueeze(0))
#                 pred_n = F.normalize(pred.squeeze(0), dim=-1)
#                 target_n = F.normalize(target.to(device), dim=-1)
#                 cos_sim = (pred_n * target_n).sum().item()
            
#             similarities.append(cos_sim)
        
#         if similarities:
#             avg_sim = np.mean(similarities)
#             # Low similarity → Predict Fail (0), High similarity → Predict Success (1)
#             pred_label = 0 if avg_sim < threshold else 1
#             results.append((pred_label, true_label, avg_sim))
    
#     # Calculate accuracy
#     correct = sum(pred == label for pred, label, _ in results)
#     total = len(results)
    
#     print(f"Tested {total} sequences")
#     print(f"Correct: {correct}, Accuracy: {correct/total:.4f}")
    
#     return results

# # Run it
# results = test_anomaly_accuracy(model, test_dataset, threshold=0.8)

Tested 200 sequences
Correct: 46, Accuracy: 0.2300


In [15]:
def test_anomaly_accuracy(model, test_dataset, threshold=0.9):
    """Test if model correctly identifies Fail vs Success sequences"""
    
    results = []
    
    for i in range(200):
        seq, label_vec = test_dataset[i]
        
        # Skip if too short
        if len(seq) < 15:
            continue
        
        # True label: 0=Fail, 1=Success
        true_label = 1 if label_vec[1] == 1 else 0
        
        similarities = []
        device = next(model.parameters()).device
        
        for pos in range(5, len(seq) - 1):
            context = seq[:pos+1]
            target = seq[pos+1]
            
            if len(context) > model.max_seq_len:
                context = context[-model.max_seq_len:]
            
            if len(context) < model.max_seq_len:
                padded = torch.zeros((model.max_seq_len, seq.shape[-1]), device=device)
                start_idx = model.max_seq_len - len(context)
                padded[start_idx:] = context.to(device)
                mask = torch.ones(model.max_seq_len, dtype=torch.bool, device=device)
                mask[start_idx:] = False
            else:
                padded = context.to(device)
                mask = torch.zeros(model.max_seq_len, dtype=torch.bool, device=device)
            
            with torch.no_grad():
                pred = model(padded.unsqueeze(0), mask.unsqueeze(0))
                pred_n = F.normalize(pred.squeeze(0), dim=-1)
                target_n = F.normalize(target.to(device), dim=-1)
                cos_sim = (pred_n * target_n).sum().item()
            
            similarities.append(cos_sim)
        
        if similarities:
            avg_sim = np.mean(similarities)
            # Low similarity → Predict Fail (0), High similarity → Predict Success (1)
            pred_label = 0 if avg_sim < threshold else 1
            results.append((pred_label, true_label, avg_sim))
            
            # PRINT DEBUG INFO
            if i < 10:  # Print first 10 sequences
                label_name = "Success" if true_label == 1 else "Fail"
                print(f"Seq {i}: {label_name}, avg_sim={avg_sim:.4f}, pred={'Fail' if pred_label==0 else 'Success'}")
    
    # Calculate accuracy
    correct = sum(pred == label for pred, label, _ in results)
    total = len(results)
    
    print(f"\nTested {total} sequences")
    print(f"Correct: {correct}, Accuracy: {correct/total:.4f}")
    
    return results

# Run it
results = test_anomaly_accuracy(model, test_dataset, threshold=0.8)

Seq 0: Success, avg_sim=0.0184, pred=Fail
Seq 1: Success, avg_sim=0.0243, pred=Fail
Seq 2: Fail, avg_sim=0.0240, pred=Fail
Seq 3: Success, avg_sim=0.0258, pred=Fail
Seq 4: Success, avg_sim=0.0150, pred=Fail
Seq 5: Success, avg_sim=0.0200, pred=Fail
Seq 6: Fail, avg_sim=0.0194, pred=Fail
Seq 7: Success, avg_sim=0.0280, pred=Fail
Seq 8: Fail, avg_sim=0.0273, pred=Fail
Seq 9: Fail, avg_sim=0.0302, pred=Fail

Tested 200 sequences
Correct: 46, Accuracy: 0.2300


In [11]:
# Test 2 sequences manually
print("TESTING SIMILARITY MANUALLY:")
print("=" * 40)

# Find one Success and one Fail sequence
for i in range(20):
    seq, label = test_dataset[i]
    if label[1] == 1:  # Success
        success_seq = seq
        success_idx = i
        print(f"Found Success at index {i}")
        break

for i in range(20):
    seq, label = test_dataset[i]
    if label[0] == 1:  # Fail
        fail_seq = seq
        fail_idx = i
        print(f"Found Fail at index {i}")
        break

# Test Success sequence
print(f"\n1. Success sequence (index {success_idx}):")
success_similarities = []
for pos in [5, 10, 15]:  # Just test 3 positions
    if pos < len(success_seq) - 1:
        context = success_seq[:pos+1]
        target = success_seq[pos+1]
        
        # Your prediction code here...
        # Calculate similarity
        sim = 0  # Replace with actual calculation
        success_similarities.append(sim)
        print(f"   Position {pos}: similarity = {sim:.3f}")

if success_similarities:
    print(f"   Average: {np.mean(success_similarities):.3f} (should be ~0.88)")

# Test Fail sequence  
print(f"\n2. Fail sequence (index {fail_idx}):")
fail_similarities = []
for pos in [5, 10, 15]:
    if pos < len(fail_seq) - 1:
        context = fail_seq[:pos+1]
        target = fail_seq[pos+1]
        
        # Same prediction code
        sim = 0  # Replace with actual
        fail_similarities.append(sim)
        print(f"   Position {pos}: similarity = {sim:.3f}")

if fail_similarities:
    print(f"   Average: {np.mean(fail_similarities):.3f} (should be ~0.47)")

TESTING SIMILARITY MANUALLY:
Found Success at index 0
Found Fail at index 2

1. Success sequence (index 0):
   Position 5: similarity = 0.000
   Position 10: similarity = 0.000
   Position 15: similarity = 0.000
   Average: 0.000 (should be ~0.88)

2. Fail sequence (index 2):
   Position 5: similarity = 0.000
   Position 10: similarity = 0.000
   Position 15: similarity = 0.000
   Average: 0.000 (should be ~0.47)
